<a href="https://colab.research.google.com/github/sayligadgil/Land-cover-classification/blob/main/01_GEE_Data_Loading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install earthengine-api geemap -q

print("✅ Libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 44.5 MB/s eta 0:00:00
✅ Libraries installed!


In [4]:
# Import and authenticate Earth Engine
import ee
import geemap

# Authenticate (browser popup will appear)
ee.Authenticate()

# Initialize Earth Engine
ee.Initialize(project='angelic-digit-490707-f3')

print("✅ Earth Engine authenticated and initialized!")
print("🚀 Ready to load satellite data!")

✅ Earth Engine authenticated and initialized!
🚀 Ready to load satellite data!


In [5]:
# Hyderabad study area
center_lat = 17.4200
center_lon = 78.4700

# Create center point
center_point = ee.Geometry.Point([center_lon, center_lat])

# Create 12.5km radius buffer (25km diameter study area)
study_area = center_point.buffer(12500)

print(f"✅ Study area defined!")
print(f"   Location: {center_lat}°N, {center_lon}°E")
print(f"   Size: ~625 sq km (25km × 25km)")
print(f"   City: Hyderabad, Telangana, India")

✅ Study area defined!
   Location: 17.42°N, 78.47°E
   Size: ~625 sq km (25km × 25km)
   City: Hyderabad, Telangana, India


In [6]:
# Create interactive map centered on Hyderabad
Map = geemap.Map(center=[center_lat, center_lon], zoom=11)

# Add study area boundary (red circle)
Map.addLayer(study_area, {'color': 'red'}, 'Study Area Boundary')

# Display the map
Map

Map(center=[17.42, 78.47], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [7]:
# Search for cloud-free Landsat 8 images
# Date range: October 2024 - February 2025 (dry season)
start_date = '2024-10-01'
end_date = '2025-02-28'

# Load Landsat 8 Collection
landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterBounds(study_area) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUD_COVER', 10))

# Count available images
count = landsat.size().getInfo()
print(f"🛰️ Found {count} Landsat 8 images with <10% cloud cover")

# Get details of the best (least cloudy) image
if count > 0:
    best_image = landsat.sort('CLOUD_COVER').first()

    image_id = best_image.get('LANDSAT_PRODUCT_ID').getInfo()
    cloud_cover = best_image.get('CLOUD_COVER').getInfo()
    date = best_image.date().format('YYYY-MM-dd').getInfo()

    print(f"\n📸 Best available image:")
    print(f"   Date: {date}")
    print(f"   Cloud cover: {cloud_cover:.2f}%")
    print(f"   Scene ID: {image_id}")
else:
    print("⚠️ No images found - try expanding date range")

🛰️ Found 6 Landsat 8 images with <10% cloud cover

📸 Best available image:
   Date: 2025-02-25
   Cloud cover: 0.13%
   Scene ID: LC08_L2SP_144048_20250225_20250304_02_T1


In [8]:
# Get the best image
best_image = landsat.sort('CLOUD_COVER').first()

# Scale Landsat Surface Reflectance values
def scale_landsat(image):
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, None, True)

# Apply scaling
scaled_image = scale_landsat(best_image)

# Clip to study area
clipped_image = scaled_image.clip(study_area)

# Create TRUE COLOR visualization (like a photograph)
rgb_viz = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],  # Red, Green, Blue
    'min': 0,
    'max': 0.3
}

# Create new map
Map2 = geemap.Map(center=[center_lat, center_lon], zoom=11)
Map2.addLayer(clipped_image, rgb_viz, 'Hyderabad - True Color')
Map2.addLayer(study_area, {'color': 'yellow'}, 'Study Area', opacity=0.3)

Map2

Map(center=[17.42, 78.47], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [9]:
# FALSE COLOR composite (highlights vegetation in RED)
false_color_viz = {
    'bands': ['SR_B5', 'SR_B4', 'SR_B3'],  # NIR, Red, Green
    'min': 0,
    'max': 0.4
}

Map3 = geemap.Map(center=[center_lat, center_lon], zoom=11)
Map3.addLayer(clipped_image, false_color_viz, 'False Color - Vegetation')
Map3

Map(center=[17.42, 78.47], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…